In [1]:
pip install torch torchvision


  Obtaining dependency information for torch from https://files.pythonhosted.org/packages/5c/01/5ab75f138bf32d7a69df61e4997e24eccad87cc009f5fb7e2a31af8a4036/torch-2.2.2-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for torchvision from https://files.pythonhosted.org/packages/c6/75/d869f600fc33df8b8ca99943e165a4ca23b73c68dc1942098fde0a6b46f3/torchvision-0.17.2-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for typing-extensions>=4.8.0 from https://files.pythonhosted.org/packages/01/f3/936e209267d6ef7510322191003885de524fc48d1b43269810cd589ceaf5/typing_extensions-4.11.0-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/198.6 MB ? eta -:--:--
   ---------------------------------------- 0.2/198.6 MB 3.6 MB/s eta 0:00:56
   ---------------------------------------- 0.4/198.6 MB 5.0 MB/s eta 0:00:40
   ---------------------------------------- 0.6/198.6 MB 5.1 MB/s eta 0:00:39
   ---------------------------------------- 0.8

In [4]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Define transformations
transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load data
train_data = datasets.ImageFolder(root='C:\\Users\\vedas\\Downloads\\archive (21)\\seg_train\\seg_train', transform=transform)
test_data = datasets.ImageFolder(root='C:\\Users\\vedas\\Downloads\\archive (21)\\seg_test\\seg_test', transform=transform)

# Data loaders
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)


In [5]:
import torch.nn as nn
import torch.nn.functional as F

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128 * 18 * 18, 512)
        self.fc2 = nn.Linear(512, 6)
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(-1, 128 * 18 * 18)
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


In [6]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_model(num_epochs):
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
        print(f'Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}')

train_model(10)


Epoch 1, Loss: 0.9979922552858236
Epoch 2, Loss: 0.6306022632366433
Epoch 3, Loss: 0.4592268255142525
Epoch 4, Loss: 0.3342599712380108
Epoch 5, Loss: 0.21773188352024772
Epoch 6, Loss: 0.14355824417424026
Epoch 7, Loss: 0.0975113188782134
Epoch 8, Loss: 0.08476367822672179
Epoch 9, Loss: 0.06272858062097145
Epoch 10, Loss: 0.06083236120210044


In [7]:
def evaluate_model():
    model.eval()
    total = 0
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f'Accuracy: {100 * correct / total}%')

evaluate_model()


Accuracy: 81.46666666666667%


In [8]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import KFold
import numpy as np

# Define transformations
transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load data
data = datasets.ImageFolder(root='C:\\Users\\vedas\\Downloads\\archive (21)', transform=transform)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import confusion_matrix

kf = KFold(n_splits=10, shuffle=True, random_state=42)
results = []
conf_matrix = np.zeros((6, 6))  # Assuming 6 classes

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for fold, (train_idx, test_idx) in enumerate(kf.split(np.arange(len(data)))):
    train_subsampler = Subset(data, train_idx)
    test_subsampler = Subset(data, test_idx)
    
    train_loader = DataLoader(train_subsampler, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_subsampler, batch_size=32, shuffle=False)
    
    model = CNN().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Train the model for this fold
    for epoch in range(10):  # Training for 10 epochs per fold
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
    # Evaluate the model
    model.eval()
    total = 0
    correct = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            all_preds.extend(predicted.view(-1).cpu().numpy())
            all_labels.extend(labels.view(-1).cpu().numpy())
    
    acc = 100 * correct / total
    results.append(acc)
    conf_matrix += confusion_matrix(all_labels, all_preds, labels=[0, 1, 2, 3, 4, 5])
    print(f'Fold {fold + 1}, Accuracy: {acc}%')

# Display overall results
print(f'Average Accuracy: {np.mean(results)}%')
print(f'Standard Deviation: {np.std(results)}%')
print("Confusion Matrix:")
print(conf_matrix)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix
import numpy as np

# Define transformations
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Load data
data = datasets.ImageFolder(root='C:\\Users\\vedas\\Downloads\\archive (21)', transform=transform)
train_loader = DataLoader(data, batch_size=32, shuffle=True)

# Load a pretrained AlexNet model
model = models.alexnet(pretrained=True)

# Freeze all parameters
for param in model.parameters():
    param.requires_grad = False

# Replace the classifier
model.classifier[6] = nn.Linear(model.classifier[6].in_features, 6)  # Assuming 6 classes

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier[6].parameters(), lr=0.001)  # Optimize only the last layer

# Train the model
model.train()
for epoch in range(3):  # Few epochs due to pretraining
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

# Evaluate the model
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Compare performance
conf_matrix = confusion_matrix(all_labels, all_preds, labels=list(range(6)))
print("Confusion Matrix:")
print(conf_matrix)
